# Pump It Up model comparison

Define the first feature policy, prove that its learned preprocessing fits within development folds and compare the first two feature-based candidates with the majority reference.

The notebook keeps the decisions visible while delegating mechanical feature engineering, preprocessing and evaluation to focused modules under `src/`. It reconstructs the frozen design from `02-baseline.ipynb`; the local test and competition rows remain outside every fit and score.

In [1]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != 'notebooks' or NOTEBOOK_DIR.parent.name != 'stage-1-pump-it-up':
    raise RuntimeError(
        'Run this notebook from the stage-1-pump-it-up/notebooks directory.'
    )

STAGE_DIR = NOTEBOOK_DIR.parent
DATA_DIR = STAGE_DIR / 'data'
SRC_DIR = STAGE_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from baseline_evaluation import evaluate_majority_reference
from data_partitioning import make_cross_validation, partition_modelling_data
from feature_engineering import summarise_initial_feature_policy
from model_evaluation import (
    compare_candidate_diversity,
    compare_candidate_evaluations,
    compare_with_majority_reference,
    evaluate_equal_weight_soft_vote,
    evaluate_gaussian_naive_bayes,
    evaluate_initial_decision_tree,
    evaluate_initial_extra_trees,
    evaluate_initial_histogram_boosting,
    evaluate_k_nearest_neighbours,
    evaluate_logistic_regression,
    evaluate_random_forest,
    summarise_classifier_screen,
    summarise_initial_decision_tree,
    summarise_initial_extra_trees,
    summarise_initial_histogram_boosting,
)
from model_preprocessing import smoke_test_first_fold
from modelling_data import prepare_modelling_data

## Reconstruct the frozen development design

Load the immutable source files, apply the settled structural preparation and recreate the fixed local test membership and five development folds. No cleaned copy is written.

In [2]:
raw_original = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
labels_original = pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv')
raw_competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')

modelling_data = prepare_modelling_data(
    raw_original,
    labels_original,
    raw_competition,
)
partitioned_data = partition_modelling_data(modelling_data)
cross_validation = make_cross_validation(partitioned_data)

## Fix the initial feature policy

`date_recorded` becomes `days_since_recorded`; it is not split into year, month or day fields. The fixed reference is **2 February 2015**, the date of the [earliest surviving official Pump It Up community post](https://community.drivendata.org/t/about-the-pump-it-up-data-mining-the-water-table-category/63). DrivenData no longer exposes a verified original launch date, so the code labels this honestly as a competition-era reference date. All supplied recording dates precede it.

The first pass also favours a compact, interpretable feature set. High-cardinality identifiers and alternate levels of deterministic category hierarchies are deferred for later ablations rather than allowed to dominate the first model.

In [3]:
print(summarise_initial_feature_policy().to_string())

                                                                                                                                                       treatment
feature_group                                                                                                                                                   
Recording date                                                                                    Replace date_recorded with days_since_recorded from 2015-02-02
Construction year                                                                         Replace with pump_age_at_recording plus missing and inconsistent flags
Measurement sentinels                                        Flag unavailable height, coordinates and population; leave fold-fitted median imputation downstream
Categorical predictors                                                               Use explicit missing values, fold-fitted rare grouping and one-hot encoding
Deferred high-cardinality         

## Smoke-test preprocessing on one fold

Use fold 1 as validation. Deterministic feature engineering runs inside the pipeline, then numeric medians, missing-value indicators, rare-category grouping and one-hot categories are learned from the other four folds only. Transform fold 1 without refitting.

The helper also sends a one-row synthetic unseen category through the fitted pipeline. This checks the fallback path even if every naturally occurring validation level happened to appear in the training rows.

In [4]:
preprocessing_smoke = smoke_test_first_fold(
    partitioned_data,
    cross_validation,
)
initial_preprocessor = preprocessing_smoke.preprocessor

print(preprocessing_smoke.summary.to_string())

unseen_validation_categories = preprocessing_smoke.categorical_coverage.query(
    'unseen_validation_levels > 0'
)
if unseen_validation_categories.empty:
    print('All selected categorical levels in fold 1 also occur in its training rows.')
else:
    print(unseen_validation_categories.to_string())

                                                 value
measurement                                           
Validation fold                                      1
Training rows                                    38016
Validation rows                                   9504
Raw predictors received                             36
Engineered predictors selected                      29
Transformed predictors                             301
Sparse output                                     True
Synthetic unseen category handled                 True
Competition-era reference date              2015-02-02
Training days_since_recorded range    426 to 4494 days
Validation days_since_recorded range  426 to 3990 days
Non-finite transformed values                        0
All selected categorical levels in fold 1 also occur in its training rows.


### Interpretation

The fold-fitted pipeline accepts all 36 prepared predictors, selects and engineers 29 initial predictors, and produces a finite sparse matrix for both training and validation rows. `date_recorded` is absent after engineering; the only direct recording-time feature is elapsed days from the fixed competition-era reference.

The local test and competition data have still not been transformed, inspected or scored.

## Evaluate the first feature-based model

Use one fixed decision tree as a transparent first candidate. Depth and leaf-size constraints limit memorisation; class weighting remains off so this run records unadjusted behaviour. The complete feature-engineering, preprocessing and classifier pipeline is fitted independently in each frozen development fold.

In [5]:
print(summarise_initial_decision_tree().to_string())

_, majority_summary = evaluate_majority_reference(
    partitioned_data,
    cross_validation,
)
tree_evaluation = evaluate_initial_decision_tree(
    partitioned_data,
    cross_validation,
)
model_comparison = compare_with_majority_reference(
    majority_summary,
    tree_evaluation,
)

                     value                                                    purpose
setting                                                                              
criterion             gini            Standard transparent multiclass split criterion
max_depth               12     Limit memorisation and keep the first tree inspectable
min_samples_leaf        20         Prevent leaves supported by only a handful of rows
class_weight          None  Measure unweighted behaviour before imbalance experiments
random_state      20260820                       Make tied split choices reproducible


## Results

Report every fold before the aggregate. Accuracy is the competition measure; per-class recall and the confusion matrix show which outcomes that single score hides.

In [6]:
as_percentages = lambda frame: frame.map(lambda value: f'{value:.2%}')

print('Fold metrics')
print(as_percentages(tree_evaluation.fold_metrics).to_string())
print('\nMean comparison')
print(as_percentages(model_comparison).to_string())
print('\nTree shapes')
print(tree_evaluation.diagnostics.to_string())
print('\nAggregate confusion counts')
print(tree_evaluation.confusion_counts.to_string())
print('\nRecall-normalised confusion matrix')
print(as_percentages(tree_evaluation.confusion_recall).to_string())

Fold metrics
                accuracy recall: functional recall: functional needs repair recall: non functional
validation_fold                                                                                   
1                 74.68%             90.18%                          14.78%                 64.10%
2                 75.12%             91.28%                          13.31%                 63.96%
3                 75.33%             91.07%                          16.50%                 64.21%
4                 74.76%             91.13%                          16.21%                 62.71%
5                 75.02%             90.27%                          15.05%                 64.81%

Mean comparison
                                majority reference decision tree absolute change
metric                                                                          
accuracy                                    54.31%        74.98%          20.67%
recall: functional                

### Interpretation

The constrained tree averages **74.98% accuracy**, a **20.67 percentage-point improvement** over the 54.31% majority reference. Fold accuracy ranges only from 74.68% to 75.33%, so this gain is not driven by one favourable validation fold.

The model recovers **90.79%** of `functional` rows and **63.96%** of `non functional` rows, but only **15.17%** of `functional needs repair`. About 70% of repair rows are still predicted as functional. All five fitted trees reach the depth-12 limit, confirming that the constraint is active rather than decorative.

This is a credible first feature-model baseline, not a selected model. The local test and competition data remain untouched.

## Evaluate Extra Trees on the same evidence

Reuse the 500-tree configuration from the early competition experiment, but replace its full-data preparation with the current fold-fitted pipeline and 29-feature policy. The folds, features and preprocessing are unchanged from the constrained tree, so this comparison isolates the practical effect of the model family.

In [7]:
print(summarise_initial_extra_trees().to_string())

extra_trees_evaluation = evaluate_initial_extra_trees(
    partitioned_data,
    cross_validation,
    record_elapsed=True,
)
model_comparison = compare_candidate_evaluations(
    majority_summary,
    tree_evaluation,
    extra_trees_evaluation,
)
model_comparison['Extra Trees change from tree'] = (
    model_comparison['Extra Trees'] - model_comparison['decision tree']
)

                     value                                                purpose
setting                                                                          
n_estimators           500           Reuse the prior early-experiment forest size
max_features           0.7               Reuse the prior feature subsampling rate
min_samples_leaf         1                 Reuse the prior unrestricted leaf size
bootstrap            False   Use the complete fold-training sample for every tree
class_weight          None    Isolate model capacity before imbalance experiments
random_state      20260815  Match the reproducible early-experiment configuration


## Extra Trees results

Use the same fold-level and aggregate evidence as the constrained tree. Forest diagnostics describe the fitted capacity; they are not tuning targets.

In [8]:
print('Extra Trees fold metrics')
print(as_percentages(extra_trees_evaluation.fold_metrics).to_string())
print('\nMean comparison')
print(as_percentages(model_comparison).to_string())
print('\nForest diagnostics')
print(extra_trees_evaluation.diagnostics.round(1).to_string())
print('\nAggregate confusion counts')
print(extra_trees_evaluation.confusion_counts.to_string())
print('\nRecall-normalised confusion matrix')
print(as_percentages(extra_trees_evaluation.confusion_recall).to_string())

Extra Trees fold metrics
                accuracy recall: functional recall: functional needs repair recall: non functional
validation_fold                                                                                   
1                 78.67%             84.15%                          41.01%                 78.04%
2                 78.86%             85.16%                          34.88%                 78.28%
3                 78.11%             83.34%                          41.24%                 77.71%
4                 79.63%             85.29%                          40.38%                 79.05%
5                 78.92%             84.79%                          39.51%                 78.09%

Mean comparison
                                majority reference decision tree Extra Trees Extra Trees change from tree
metric                                                                                                   
accuracy                                    54.31%   

### Interpretation

Extra Trees averages **78.84% accuracy**, improving on the constrained tree by **3.86 percentage points**. Fold accuracy ranges from 78.11% to 79.63%, so the gain appears across the frozen development design rather than in a single fold.

The forest trades some dominant-class performance for much stronger separation elsewhere. `functional` recall falls by 6.24 points to **84.55%**, while `functional needs repair` recall rises by 24.23 points to **39.40%** and `non functional` recall rises by 14.28 points to **78.24%**. Repair remains the weakest class, but it is no longer almost entirely absorbed into `functional`.

This local result is 1.66 points below the earlier 80.50% public Extra Trees score. The evaluations and feature preparation differ, so that is not a direct performance gap, but the closeness supports the earlier experiment as useful model-family evidence. The cost is material: the five-fold run took about 18 minutes on this machine, and the fitted trees average roughly 65 levels and 12,200 leaves. Extra Trees is the leading local candidate, not yet the selected final workflow. The local test and competition data remain untouched.

## Evaluate histogram gradient boosting

Reuse the earlier boosting configuration with dense output from the same fold-fitted one-hot preprocessor. Dense and sparse preprocessing produce identical 301 feature values on the first frozen fold; the representation changes only because scikit-learn's histogram booster requires dense input. Record every fold's elapsed time as part of the comparison.

In [9]:
print(summarise_initial_histogram_boosting().to_string())

histogram_boosting_evaluation = evaluate_initial_histogram_boosting(
    partitioned_data,
    cross_validation,
)
all_model_comparison = compare_candidate_evaluations(
    majority_summary,
    tree_evaluation,
    extra_trees_evaluation,
    histogram_boosting_evaluation,
)
all_model_comparison['boosting change from Extra Trees'] = (
    all_model_comparison['histogram boosting']
    - all_model_comparison['Extra Trees']
)

                      value                                                purpose
setting                                                                           
learning_rate          0.08        Reuse the prior early-experiment shrinkage rate
max_iter                300          Reuse the prior number of boosting iterations
max_leaf_nodes           31              Reuse the prior per-tree complexity limit
min_samples_leaf         20                   Reuse the prior minimum leaf support
l2_regularization       1.0              Reuse the prior leaf-value regularisation
max_features            0.8               Reuse the prior feature subsampling rate
early_stopping        False     Keep every fold on the same fixed iteration budget
random_state       20260815  Match the reproducible early-experiment configuration
Completed histogram boosting fold 1/5 in 27.9 seconds.
Completed histogram boosting fold 2/5 in 39.6 seconds.
Completed histogram boosting fold 3/5 in 32.1 seconds.
Compl

## Histogram boosting results

Compare predictive evidence and computational cost. Accuracy remains the competition measure; per-class recall determines what that gain trades between outcomes.

In [10]:
print('Histogram boosting fold metrics')
print(as_percentages(histogram_boosting_evaluation.fold_metrics).to_string())
print('\nMean comparison')
print(as_percentages(all_model_comparison).to_string())
print('\nBoosting diagnostics')
print(histogram_boosting_evaluation.diagnostics.round(1).to_string())
print('\nTiming totals')
timing_columns = ['fit_seconds', 'predict_seconds', 'total_seconds']
print(
    histogram_boosting_evaluation.diagnostics[timing_columns]
    .sum()
    .round(1)
    .to_string()
)
print('\nAggregate confusion counts')
print(histogram_boosting_evaluation.confusion_counts.to_string())
print('\nRecall-normalised confusion matrix')
print(
    as_percentages(histogram_boosting_evaluation.confusion_recall)
    .to_string()
)

Histogram boosting fold metrics
                accuracy recall: functional recall: functional needs repair recall: non functional
validation_fold                                                                                   
1                 80.05%             90.08%                          30.14%                 75.30%
2                 80.12%             90.62%                          25.18%                 75.68%
3                 79.95%             89.50%                          29.81%                 75.93%
4                 80.77%             90.89%                          31.40%                 75.79%
5                 80.16%             90.16%                          26.92%                 76.10%

Mean comparison
                                majority reference decision tree Extra Trees histogram boosting boosting change from Extra Trees
metric                                                                                                                          


### Interpretation

Histogram boosting averages **80.21% accuracy**, improving on Extra Trees by **1.37 percentage points** and on the constrained tree by 5.23 points. Fold accuracy stays between 79.95% and 80.77%, making it the strongest and most stable local accuracy result so far.

The gain has a class trade-off. Compared with Extra Trees, boosting raises `functional` recall by 5.70 points to **90.25%**, but lowers `functional needs repair` recall by 10.71 points to **28.69%** and `non functional` recall by 2.48 points to **75.76%**. Extra Trees therefore remains the better repair detector, while boosting leads on the stated competition metric.

The five folds complete in **179.5 seconds**, about three minutes and roughly six times faster than the 18-minute Extra Trees run on this machine. The local 80.21% result is 0.53 points above the earlier 79.68% public boosting score, but their evaluation designs differ. Histogram boosting is now the leading local candidate by accuracy; the local test and competition data remain untouched.

## Evaluate the equal-weight soft vote

Average the aligned out-of-fold class probabilities from Extra Trees and histogram boosting. This reproduces the early experiment's equal-weight rule without fitting on validation rows or choosing weights from the result.

In [11]:
soft_vote_evaluation = evaluate_equal_weight_soft_vote(
    partitioned_data,
    cross_validation,
    extra_trees_evaluation,
    histogram_boosting_evaluation,
)
ensemble_comparison = compare_candidate_evaluations(
    majority_summary,
    tree_evaluation,
    extra_trees_evaluation,
    histogram_boosting_evaluation,
    soft_vote_evaluation,
)
ensemble_comparison['soft vote change from boosting'] = (
    ensemble_comparison['soft vote']
    - ensemble_comparison['histogram boosting']
)

## Soft-vote results

The blend has no additional learned parameters. Its cost is the combined fitting and probability-prediction cost of the two component workflows.

In [12]:
print('Soft-vote fold metrics')
print(as_percentages(soft_vote_evaluation.fold_metrics).to_string())
print('\nMean comparison')
print(as_percentages(ensemble_comparison).to_string())
print('\nBlend weights')
weight_columns = ['Extra Trees weight', 'histogram boosting weight']
print(soft_vote_evaluation.diagnostics[weight_columns].to_string())
print('\nAggregate confusion counts')
print(soft_vote_evaluation.confusion_counts.to_string())
print('\nRecall-normalised confusion matrix')
print(as_percentages(soft_vote_evaluation.confusion_recall).to_string())

Soft-vote fold metrics
                accuracy recall: functional recall: functional needs repair recall: non functional
validation_fold                                                                                   
1                 81.00%             88.30%                          38.12%                 78.78%
2                 80.78%             88.80%                          34.30%                 78.23%
3                 80.17%             87.27%                          36.76%                 78.34%
4                 81.51%             88.88%                          37.34%                 79.46%
5                 80.70%             88.34%                          35.46%                 78.48%

Mean comparison
                                majority reference decision tree Extra Trees histogram boosting soft vote soft vote change from boosting
metric                                                                                                                            

### Interpretation

The equal-weight soft vote averages **80.83% accuracy**, improving on histogram boosting by **0.62 percentage points** and on Extra Trees by 1.99 points. It beats boosting in every validation fold, with fold accuracy between 80.17% and 81.51%.

Compared with boosting, the blend lowers `functional` recall by 1.93 points to **88.32%**, while raising `functional needs repair` recall by 7.70 points to **36.39%** and `non functional` recall by 2.90 points to **78.66%**. It therefore recovers much of Extra Trees' minority-class strength without surrendering the accuracy lead. Extra Trees alone still has the highest repair recall at 39.40%.

Refitting both components takes about **20.4 minutes** on this machine; the probability average itself is negligible. The local result is 0.87 points below the earlier 81.70% public soft-vote score, but the evaluation and feature preparation differ. The equal-weight blend is now the leading development-fold workflow. The local test and competition data remain untouched.

## Broaden the classifier screen

Add four deliberately different classifier families. Logistic regression tests a mostly additive decision boundary. KNN tests local similarity and is included even though one-hot data may make distance less informative. Gaussian naïve Bayes supplies a very fast model with deliberately strong distribution and independence assumptions. Random Forest tests conventional bootstrap aggregation against Extra Trees' greater split randomisation.

Numeric predictors are standardised inside each training fold for the linear, neighbour and Gaussian models. Categorical encoding and all other learned preprocessing remain fold-fitted. The local test and competition rows remain untouched.

In [13]:
print(summarise_classifier_screen().to_string())

logistic_evaluation = evaluate_logistic_regression(
    partitioned_data,
    cross_validation,
)
gaussian_nb_evaluation = evaluate_gaussian_naive_bayes(
    partitioned_data,
    cross_validation,
)
knn_evaluation = evaluate_k_nearest_neighbours(
    partitioned_data,
    cross_validation,
)
random_forest_evaluation = evaluate_random_forest(
    partitioned_data,
    cross_validation,
)

                             family                              key_setting                                                  reason
method                                                                                                                              
logistic regression          linear                 L2 regularisation, C=1.0     Test whether mostly additive effects are sufficient
KNN                       neighbour  k=25, distance weighted, scaled numeric        Show the effect of a local distance-based method
Gaussian naïve Bayes  probabilistic                      var_smoothing=1e-09  Test a fast model with strong independence assumptions
Random Forest          bagged trees         300 trees, sqrt feature sampling           Compare conventional bagging with Extra Trees
Completed logistic regression fold 1/5 in 2.1 seconds.
Completed logistic regression fold 2/5 in 2.2 seconds.
Completed logistic regression fold 3/5 in 2.5 seconds.
Completed logistic regression fold 4/

## Broad-screen results

Compare every single model on the same five held-out folds. Runtime is diagnostic rather than a scoring metric, but it helps distinguish methods suitable for repeated improvement rounds from expensive final candidates.

In [14]:
screen_evaluations = (
    logistic_evaluation,
    gaussian_nb_evaluation,
    knn_evaluation,
    random_forest_evaluation,
)
single_model_comparison = compare_candidate_evaluations(
    majority_summary,
    tree_evaluation,
    logistic_evaluation,
    gaussian_nb_evaluation,
    knn_evaluation,
    extra_trees_evaluation,
    histogram_boosting_evaluation,
    random_forest_evaluation,
)
new_fold_accuracy = pd.DataFrame({
    evaluation.model_name: evaluation.fold_metrics['accuracy']
    for evaluation in screen_evaluations
})
runtime_seconds = pd.Series(
    {
        evaluation.model_name: evaluation.diagnostics[
            'total_seconds'
        ].sum()
        for evaluation in screen_evaluations
    },
    name='five-fold seconds',
)

print('Single-model mean comparison')
print(as_percentages(single_model_comparison).to_string())
print('\nNew candidates: fold accuracy')
print(as_percentages(new_fold_accuracy).to_string())
print('\nNew candidates: total runtime')
print(runtime_seconds.round(1).to_string())

Single-model mean comparison
                                majority reference decision tree logistic regression Gaussian naïve Bayes     KNN Extra Trees histogram boosting Random Forest
metric                                                                                                                                                        
accuracy                                    54.31%        74.98%              75.06%               28.33%  77.89%      78.84%             80.21%        80.59%
recall: functional                         100.00%        90.79%              88.69%               21.43%  87.12%      84.55%             90.25%        87.64%
recall: functional needs repair              0.00%        15.17%              13.96%               94.67%  31.99%      39.40%             28.69%        36.86%
recall: non functional                       0.00%        63.96%              67.35%               25.53%  73.51%      78.24%             75.76%        78.90%

New candidates: 

### Interpretation

Random Forest is the strongest new single model at **80.59% mean accuracy**. It sits only 0.24 percentage points below the existing two-model soft vote, while recording **36.86% repair recall** and **78.90% non-functional recall**. It therefore enters the ensemble round.

KNN reaches a credible **77.89%** rather than failing on the wide one-hot representation. Logistic regression reaches **75.06%**, close to the constrained tree, showing the limit of a mostly additive boundary here. Gaussian naïve Bayes demonstrates the sharpest method-selection effect: only **28.33% accuracy**, but **94.67% repair recall** because it over-predicts the rare class. That is interesting diagnostic behaviour, not a sound ensemble component without separate calibration work.

The four additions take about three minutes in total across all five folds on this machine. KNN is prediction-heavy but manageable; Random Forest accounts for most of the added cost.

## Run a small ensemble round

Accuracy alone does not determine ensemble membership. First compare where the four credible non-linear candidates disagree and where only one model is correct. Then test a small set of simple equal-weight blends declared in advance: the two forests, Random Forest plus boosting, all three tree ensembles, and those three plus KNN. This is a bounded comparison, not an exhaustive search for weights on the validation results.

In [15]:
shortlist = (
    extra_trees_evaluation,
    histogram_boosting_evaluation,
    knn_evaluation,
    random_forest_evaluation,
)
shortlist_diversity = compare_candidate_diversity(
    partitioned_data,
    *shortlist,
)
forest_pair_vote = evaluate_equal_weight_soft_vote(
    partitioned_data,
    cross_validation,
    extra_trees_evaluation,
    random_forest_evaluation,
    model_name='forest-pair vote',
)
forest_boost_vote = evaluate_equal_weight_soft_vote(
    partitioned_data,
    cross_validation,
    random_forest_evaluation,
    histogram_boosting_evaluation,
    model_name='forest + boosting vote',
)
three_model_vote = evaluate_equal_weight_soft_vote(
    partitioned_data,
    cross_validation,
    extra_trees_evaluation,
    random_forest_evaluation,
    histogram_boosting_evaluation,
    model_name='three-model vote',
)
four_model_vote = evaluate_equal_weight_soft_vote(
    partitioned_data,
    cross_validation,
    extra_trees_evaluation,
    random_forest_evaluation,
    histogram_boosting_evaluation,
    knn_evaluation,
    model_name='four-model vote',
)

## Ensemble-round results

The component models have already been fitted for cross-validation. These comparisons only combine their aligned out-of-fold probabilities.

In [16]:
ensemble_round = compare_candidate_evaluations(
    majority_summary,
    extra_trees_evaluation,
    histogram_boosting_evaluation,
    knn_evaluation,
    random_forest_evaluation,
    soft_vote_evaluation,
    forest_pair_vote,
    forest_boost_vote,
    three_model_vote,
    four_model_vote,
)
print('Shortlist pairwise diversity')
print(as_percentages(shortlist_diversity).to_string())
print('\nEnsemble comparison')
print(as_percentages(ensemble_round).to_string())

Shortlist pairwise diversity
                                      disagreement left_only_correct right_only_correct both_wrong
left_model         right_model                                                                    
Extra Trees        histogram boosting       14.89%             6.09%              7.46%     13.70%
                   KNN                      12.77%             6.31%              5.35%     15.81%
                   Random Forest             6.17%             1.91%              3.67%     17.49%
histogram boosting KNN                      12.61%             6.93%              4.60%     15.19%
                   Random Forest            11.32%             4.96%              5.35%     14.45%
KNN                Random Forest            10.29%             3.36%              6.06%     16.05%

Ensemble comparison
                                majority reference Extra Trees histogram boosting     KNN Random Forest soft vote forest-pair vote forest + boosting vote thre

### Interpretation and next step

The equal-weight Random Forest and histogram-boosting vote becomes the development-fold leader at **81.37% accuracy**, improving on the earlier Extra Trees and boosting vote by **0.54 percentage points** and on Random Forest alone by 0.78 points. It records **90.03% functional recall**, **34.28% repair recall** and **78.03% non-functional recall**.

The two components disagree on 11.32% of development rows. Boosting alone is correct on 4.96% of rows where Random Forest is wrong, while Random Forest alone is correct on 5.35% where boosting is wrong. This balanced complementarity explains why the blend helps. Equal inclusion of Extra Trees or KNN lowers accuracy, so a larger ensemble is not automatically a better one.

Freeze the two-model vote for a single local-test evaluation in `05-final-model-and-submission.ipynb`, then refit it unchanged on all labelled original rows and classify the competition rows. Grouped geographic sensitivity remains an outstanding robustness check.